# AI Research Workflow · 3일차

**AI와 함께하는 연구데이터 분석** (2026-11-23)

- 위에서부터 차례로 실행합니다. 셀을 고른 뒤 Shift + Enter 를 누르면 됩니다.
- 예제 데이터는 교육용으로 만든 **가상 데이터**입니다. 실제 시험 결과가 아니므로 논문 · 보고서에 인용하지 않습니다.
- 이 노트북은 사본으로 열립니다. 고친 내용을 남기려면 「파일 → Drive에 사본 저장」을 누릅니다.
- 설명과 프롬프트는 학습 사이트 3일차 본문에 있습니다.

## 질문을 분석 계획으로

### 코드 1. 데이터를 읽고 구조 보기

설명서에는 24셀 × 11시점 = 264행이라고 적혀 있습니다. 출력이 그와 다르면 그 차이가 첫 번째 정제 대상입니다.

In [ ]:
# Google Colab 새 노트에 붙여 넣고 실행합니다.
import pandas as pd

# 교육용 가상 데이터(리튬이온 셀 24개의 사이클 시험을 흉내 낸 것)
URL = "https://raw.githubusercontent.com/jungaistar/materials/main/grad/battery_cycling_demo.csv"
df = pd.read_csv(URL)

print(df.shape)       # (행 수, 열 수). 설명서와 같은지 먼저 봅니다
print(df.head(8))     # 앞 8행을 눈으로 확인
print(df.dtypes)      # 열마다 자료형. 숫자여야 할 열이 문자면 표기가 섞였다는 신호
df.info()             # 열마다 비어 있지 않은 값의 개수

# 분석 단위 확인: 셀은 몇 개이고, 셀마다 몇 번 쟀는가
print(df["cell_id"].nunique())
print(df.groupby("cell_id").size().value_counts())

## 정제와 탐색

### 코드 2. 문제 진단

describe()의 min과 max만 봐도 단위 혼용과 이상값의 흔적이 보입니다. 용량이 수천으로 찍힌 셀과, 유지율이 한 자릿수인 행을 찾아보세요.

In [ ]:
# 코드 1에 이어서 실행합니다. 아직 아무것도 고치지 않고 세기만 합니다.

# 1) 중복 행: 모든 열이 같은 행
print("duplicated rows:", df.duplicated().sum())

# 2) 결측: 열마다 빈칸 개수
print(df.isna().sum())

# 3) 표기 혼용: 범주형 열의 고유값을 그대로 출력(공백이 보이도록 repr 사용)
for col in ["lot", "temp_c", "c_rate"]:
    print(col, [repr(v) for v in df[col].unique()])

# 4) 단위 혼용 · 이상값: 숫자 열의 범위
print(df[["capacity_ah", "retention_pct", "ir_mohm", "max_temp_c"]].describe())

# 5) 셀별로 보면 어느 셀이 문제인지 드러납니다
print(df.groupby("cell_id")["capacity_ah"].mean().sort_values().tail(3))
print(df[df["retention_pct"] < 50])

### 코드 3. 정제하고, 고친 내용을 세기

이상값을 평균에서 멀다는 이유로 지우지 않았습니다. 같은 행의 용량으로 유지율을 되짚어 계산했더니 기록된 값만 틀렸다는 근거가 있어서 고친 것입니다.

In [ ]:
# 원본은 그대로 두고 사본에서 고칩니다.
clean = df.copy()

# 1) 중복 행 제거: 모든 열이 같은 행은 하나만 남깁니다
before = len(clean)
clean = clean.drop_duplicates().reset_index(drop=True)
print("removed duplicates:", before - len(clean))

# 2) temp_c 표기 통일: "45C", " 25" 같은 값을 정수로
raw_temp = clean["temp_c"].astype(str)
clean["temp_c"] = raw_temp.str.strip().str.replace("C", "", regex=False).astype(int)
print("fixed temp labels:", (raw_temp != clean["temp_c"].astype(str)).sum())

# 3) 단위 통일: 100보다 큰 용량은 mAh로 보고 Ah로 환산
is_mah = clean["capacity_ah"] > 100
clean.loc[is_mah, "capacity_ah"] = clean.loc[is_mah, "capacity_ah"] / 1000
print("converted mAh rows:", is_mah.sum())

# 4) 이상값 확인: 유지율을 용량에서 다시 계산해 기록된 값과 비교
cap0 = clean[clean["cycle"] == 0].set_index("cell_id")["capacity_ah"]   # 셀별 0사이클 용량
recalc = clean["capacity_ah"] / clean["cell_id"].map(cap0) * 100
gap = (clean["retention_pct"] - recalc).abs()
suspect = gap > 1                      # 1%p 넘게 다르면 입력 실수로 의심
print(clean.loc[suspect, ["cell_id", "cycle", "capacity_ah", "retention_pct"]])
clean.loc[suspect, "retention_pct"] = recalc[suspect].round(2)
print("corrected outliers:", suspect.sum())

# 5) 결측은 채우지 않습니다. 몇 칸인지만 남깁니다
print("missing cells:", clean.isna().sum().sum())

# 마지막 점검: 기대한 모양과 같은가
assert len(clean) == 264
assert clean["cell_id"].nunique() == 24
assert (clean.groupby("cell_id").size() == 11).all()
print(sorted(clean["temp_c"].unique()), clean["retention_pct"].min())

### 코드 4. 조건별 요약, 상관, 탐색용 그림

그림의 글자는 영어로 적었습니다. Colab 기본 글꼴에는 한글이 없어 한글 제목이 네모로 깨지기 때문입니다. 논문용 그림은 어차피 영문 표기가 많습니다.

In [ ]:
import matplotlib.pyplot as plt

# 1) 조건별 요약: 500사이클 시점만 골라 온도 x 충전율로 묶습니다
last = clean[clean["cycle"] == 500]
summary = (last.groupby(["temp_c", "c_rate"])["retention_pct"]
               .agg(n="count", mean="mean", sd="std", min="min", max="max")
               .round(2))
print(summary)

# 2) 관계: 500사이클 시점의 측정값끼리 상관(피어슨)
print(last[["retention_pct", "ir_mohm", "max_temp_c"]].corr().round(2))

# 3) 변화: 셀마다 선 하나. 평균을 내기 전에 개별 궤적을 먼저 봅니다
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, temp in zip(axes, [25, 45]):
    part = clean[clean["temp_c"] == temp]
    for cell, g in part.groupby("cell_id"):
        style = "-" if g["c_rate"].iloc[0] == 0.5 else "--"   # 충전율은 선 모양으로 구분
        ax.plot(g["cycle"], g["retention_pct"], style, color="gray", alpha=0.7)
    ax.set_title(str(temp) + " degC (solid: 0.5C, dashed: 1.0C)")
    ax.set_xlabel("Cycle number")
axes[0].set_ylabel("Capacity retention (%)")
plt.tight_layout()
plt.show()

## 분석 방법과 시각화

### 코드 5. 셀 단위로 요약해 비교하기

기울기를 직선으로 구한 것은 요약을 위한 단순화입니다. 실제 열화 곡선이 직선이 아니면 기울기의 뜻이 달라지므로, 코드 4의 선 그림으로 모양을 먼저 확인합니다. 온도와 충전율의 상호작용까지 보려면 이원 분산분석이 후보이고, 그 설계와 해석은 상담을 받는 편이 안전합니다.

In [ ]:
import numpy as np
from scipy import stats

# 1) 셀마다 값 하나: 500사이클 용량유지율
last = clean[clean["cycle"] == 500]
g25 = last.loc[last["temp_c"] == 25, "retention_pct"]
g45 = last.loc[last["temp_c"] == 45, "retention_pct"]
print("n:", len(g25), len(g45))               # 집단별 셀 수부터 확인

# 2) Welch t 검정: 두 집단의 분산이 같다고 두지 않습니다
res = stats.ttest_ind(g25, g45, equal_var=False)
ci = res.confidence_interval(0.95)            # 평균 차이(25 - 45)의 95% 신뢰구간
print("mean diff (%p):", round(g25.mean() - g45.mean(), 2))
print("95% CI:", round(ci.low, 2), "to", round(ci.high, 2))
print("t =", round(res.statistic, 2), ", p =", res.pvalue)

# 3) 효과크기: Cohen의 d (합동 표준편차 기준)
pooled_sd = np.sqrt((g25.var(ddof=1) + g45.var(ddof=1)) / 2)
print("Cohen d:", round((g25.mean() - g45.mean()) / pooled_sd, 2))

# 4) 시간에 따른 변화: 셀마다 직선 기울기(100사이클당 %p)를 구해 온도별로 비교
def fade_slope(g):
    slope = np.polyfit(g["cycle"], g["retention_pct"], 1)[0]   # 1차 직선의 기울기
    return slope * 100

slopes = (clean.groupby(["cell_id", "temp_c"])[["cycle", "retention_pct"]]
               .apply(fade_slope).rename("slope").reset_index())
print(slopes.groupby("temp_c")["slope"].agg(["count", "mean", "std"]).round(2))
s25 = slopes.loc[slopes["temp_c"] == 25, "slope"]
s45 = slopes.loc[slopes["temp_c"] == 45, "slope"]
print(stats.ttest_ind(s25, s45, equal_var=False))

### 코드 6. 논문용 그림으로 다듬어 저장하기

캡션에 적을 것: 오차막대는 표준편차, 각 점은 셀 12개의 평균, 데이터는 교육용 가상 데이터. Colab 왼쪽 파일 목록에서 저장된 그림을 내려받을 수 있습니다.

In [ ]:
import matplotlib.pyplot as plt

# 조건(온도)별, 사이클별 평균과 표준편차. 셀 12개의 퍼짐을 오차막대로 보입니다
stat = (clean.groupby(["temp_c", "cycle"])["retention_pct"]
             .agg(mean="mean", sd="std", n="count").reset_index())

# 색각이 달라도 구분되는 두 색(Okabe-Ito 팔레트) + 표식 모양도 다르게
style = {25: dict(color="#0072B2", marker="o", label="25 degC"),
         45: dict(color="#D55E00", marker="s", label="45 degC")}

fig, ax = plt.subplots(figsize=(4.5, 3.4))          # 논문 한 단 폭에 가까운 크기(인치)
for temp, s in style.items():
    part = stat[stat["temp_c"] == temp]
    ax.errorbar(part["cycle"], part["mean"], yerr=part["sd"],
                capsize=3, linewidth=1.5, markersize=5, **s)

ax.set_xlabel("Cycle number")
ax.set_ylabel("Capacity retention (%)")
ax.set_ylim(75, 101)
ax.legend(title="Test temperature", frameon=False)
ax.spines[["top", "right"]].set_visible(False)       # 위 · 오른쪽 테두리 제거
fig.tight_layout()
fig.savefig("fig1_retention_by_temp.png", dpi=300)   # 300dpi PNG
fig.savefig("fig1_retention_by_temp.pdf")            # 벡터 형식
plt.show()
print(stat["n"].unique())                            # 각 점이 셀 몇 개의 평균인지 확인